In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

In [2]:
df = pd.read_excel("data/CleanSecurityData.xlsx")

df["Date"] = pd.to_datetime(df["date"])
df = df.sort_values(["cusip", "Date"])


In [3]:
IQR = df['spread'].describe()['75%'] - df['spread'].describe()['25%']
L = df['spread'].describe()['25%'] - 1.5 * IQR
U = df['spread'].describe()['75%'] + 1.5 * IQR

df3 = df[(df['spread'] > L) & (df['spread'] < U)]


df3["abs_spread_chg"] = df3.groupby("cusip")["spread"].diff()
df3["pct_spread_chg"] = df3["abs_spread_chg"] / df3["spread"]
df3 = df3.dropna()
df3["abs_vol"] = df3.groupby("cusip")["abs_spread_chg"].rolling(36).std().reset_index(0, drop=True)
df3["pct_vol"] = df3.groupby("cusip")["pct_spread_chg"].rolling(36).std().reset_index(0, drop=True)
df3 = df3.dropna()

df3["excess_return_vol"] = df3["dts"] * df3["pct_vol"]



b1 = df3["spread"].quantile(0.33)
b2 = df3["spread"].quantile(0.66)

df3["bucket"] = np.where(df3["spread"] < b1, "0",
                        np.where(df3["spread"] < b2, "1", "2"))

grouped = df3.groupby(["Date","bucket"]).agg(
    abs_vol_mean=("abs_vol","mean"),
    count=("cusip","count"),
    DTS_mean=("dts","mean"),
    excess_vol_mean=("excess_return_vol","mean")
).reset_index()

b0 = grouped[(grouped["bucket"] == "0") & (grouped["count"] > 5)]
b1 = grouped[(grouped["bucket"] == "1") & (grouped["count"] > 5)]
b2 = grouped[(grouped["bucket"] == "2") & (grouped["count"] > 5)]

combined = pd.concat([b0,b1,b2])

C:\Users\rycba\AppData\Local\Temp\ipykernel_82792\3105171671.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3["abs_spread_chg"] = df3.groupby("cusip")["spread"].diff()
C:\Users\rycba\AppData\Local\Temp\ipykernel_82792\3105171671.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3["pct_spread_chg"] = df3["abs_spread_chg"] / df3["spread"]
